# Prompt 12: DataFrame Creation, Column Selection and Renaming
## Databricks Certified Associate Developer for Apache Spark
### Topic 3 — DataFrame and Column Operations (20%)

---

## Ways to Create a DataFrame

| Method | Source | Typical use |
|--------|--------|-------------|
| `spark.createDataFrame(data, schema)` | Python list / pandas DF / RDD | Tests, small datasets |
| `spark.range(start, end, step)` | Generates sequential integers | Synthetic data, benchmarks |
| `spark.read.format(...).load(path)` | Files (Parquet, CSV, JSON, Delta) | Production reads |
| `rdd.toDF(column_names)` | Existing RDD | RDD → DataFrame upgrade |
| `df.toDF(*new_column_names)` | Existing DataFrame | Bulk rename all columns |
| `spark.sql('SELECT ...')` | Registered temp view / catalog | SQL-first workflows |
| `spark.table('table_name')` | Hive/Delta catalog table | Metastore tables |

---

## Schema Definition

```python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType

# Explicit schema (recommended for production)
schema = StructType([
    StructField('id',       IntegerType(), nullable=False),
    StructField('name',     StringType(),  nullable=True),
    StructField('salary',   DoubleType(),  nullable=True),
    StructField('active',   BooleanType(), nullable=True),
])

# DDL string schema (shorter, same result)
schema_ddl = 'id INT NOT NULL, name STRING, salary DOUBLE, active BOOLEAN'

# Inferred schema (convenient for exploration, avoid in production)
df = spark.read.option('inferSchema', 'true').csv('/data/file.csv')
```

**Exam tip:** Explicit schemas prevent full data scan for type inference and guarantee correct types.

In [ ]:
# Cell 1: Setup and DataFrame creation methods
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType, LongType
import pyspark.sql.functions as F
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName('DataFrameCreation') \
    .master('local[*]') \
    .getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# --- Method 1: createDataFrame with list of tuples + column names ---
data = [
    (1,  'Alice',   'Engineering', 95000.0, True),
    (2,  'Bob',     'Marketing',   72000.0, True),
    (3,  'Carol',   'Engineering', 110000.0, False),
    (4,  'Dave',    'HR',           65000.0, True),
    (5,  'Eve',     'Engineering', 105000.0, True),
]
df_names = spark.createDataFrame(data, ['id', 'name', 'department', 'salary', 'active'])
print('Method 1: createDataFrame with column names')
df_names.printSchema()
df_names.show()

# --- Method 2: createDataFrame with explicit StructType schema ---
schema = StructType([
    StructField('id',         IntegerType(), nullable=False),
    StructField('name',       StringType(),  nullable=True),
    StructField('department', StringType(),  nullable=True),
    StructField('salary',     DoubleType(),  nullable=True),
    StructField('active',     BooleanType(), nullable=True),
])
df_schema = spark.createDataFrame(data, schema)
print('Method 2: createDataFrame with StructType schema')
df_schema.printSchema()

# --- Method 3: createDataFrame with DDL string schema ---
df_ddl = spark.createDataFrame(data,
    'id INT, name STRING, department STRING, salary DOUBLE, active BOOLEAN')
print('Method 3: createDataFrame with DDL string')
df_ddl.printSchema()

# --- Method 4: spark.range() ---
df_range = spark.range(1, 11, 2)  # 1,3,5,7,9
print('Method 4: spark.range(1, 11, 2)')
df_range.show()
# range() creates a single column named 'id' of LongType
df_range.printSchema()

In [ ]:
# Cell 2: RDD-based creation — rdd.toDF() and spark.createDataFrame(rdd)

sc = spark.sparkContext

# Create RDD from list
rdd = sc.parallelize([
    (10, 'Frank',  'Finance',  88000.0),
    (11, 'Grace',  'Finance',  92000.0),
    (12, 'Henry',  'Legal',    78000.0),
])

# Option A: rdd.toDF() with column names
df_from_rdd_a = rdd.toDF(['id', 'name', 'department', 'salary'])
print('RDD → toDF() with column names')
df_from_rdd_a.printSchema()
df_from_rdd_a.show()

# Option B: spark.createDataFrame(rdd, schema)
schema_rdd = StructType([
    StructField('id',         IntegerType(), nullable=True),
    StructField('name',       StringType(),  nullable=True),
    StructField('department', StringType(),  nullable=True),
    StructField('salary',     DoubleType(),  nullable=True),
])
df_from_rdd_b = spark.createDataFrame(rdd, schema_rdd)
print('RDD → createDataFrame() with schema')
df_from_rdd_b.printSchema()

# toDF() for bulk rename — replaces ALL column names
df_renamed_all = df_names.toDF('emp_id', 'full_name', 'dept', 'annual_salary', 'is_active')
print('df.toDF() — bulk rename all columns')
df_renamed_all.printSchema()
df_renamed_all.show(3)

## Column Selection — Four Equivalent Ways

All four methods below are equivalent — they select the same columns.
The exam tests that you know all four and their subtle differences.

| Method | Syntax | Notes |
|--------|--------|-------|
| String column name | `df.select('name', 'salary')` | Simplest — can't do expressions |
| `col()` function | `df.select(col('name'), col('salary'))` | Recommended — works with expressions |
| `df[col_name]` subscript | `df.select(df['name'], df['salary'])` | DataFrame-bound — safer for joins with same column names |
| `df.col_name` attribute | `df.select(df.name, df.salary)` | Shortest but fails for columns with spaces or reserved names |

**When to use `df[col]` vs `col()`:**
- After a **join**, two DataFrames may have columns with the **same name** (e.g., both have `id`)
- `col('id')` is **ambiguous** — Spark raises `AnalysisException`
- `df_left['id']` or `df_right['id']` is **unambiguous** — explicitly references one DataFrame's column

```python
# Ambiguous — will fail:
joined.select(col('id'))  # which id? AnalysisException!

# Unambiguous — works:
joined.select(orders['id'], customers['id'].alias('customer_id'))
```

## Backtick Escaping for Special Column Names

Column names with **spaces, dots, or special characters** must be escaped with backticks in expressions:

```python
df.select(col('`first name`'))          # space in name
df.select(col('`account.balance`'))     # dot in name
df.selectExpr('`first name` AS fname')  # backtick in SQL string
spark.sql('SELECT `first name` FROM tbl')
```

In [ ]:
# Cell 3: Four ways to select columns

print('=== String column names ===')
df_names.select('name', 'salary', 'department').show(3)

print('=== col() function ===')
df_names.select(col('name'), col('salary'), col('department')).show(3)

print('=== df[col] subscript ===')
df_names.select(df_names['name'], df_names['salary'], df_names['department']).show(3)

print('=== df.col_name attribute ===')
df_names.select(df_names.name, df_names.salary, df_names.department).show(3)

# Expressions in select — col() and df[] support expressions; strings do not
print('=== Expressions in select (col() supports this; strings do not) ===')
df_names.select(
    col('name'),
    (col('salary') * 1.1).alias('salary_with_raise'),
    col('department').alias('dept')
).show(3)

# Special characters — backtick escaping
df_special = spark.createDataFrame([
    (1, 'Alice', 95000.0, 'London')
], ['id', 'first name', 'account.balance', 'city'])

print('=== Backtick escaping for special column names ===')
df_special.select(col('`first name`'), col('`account.balance`')).show()
df_special.selectExpr('`first name` AS fname', '`account.balance` AS balance').show()

## Column Renaming Methods

| Method | Renames | Notes |
|--------|---------|-------|
| `df.withColumnRenamed('old', 'new')` | Single column | Non-destructive; all other columns kept; **lazy** |
| `col('old').alias('new')` | Column in select | Used inside `select()` |
| `df.selectExpr('old AS new')` | Column in SQL expr | SQL string syntax |
| `df.toDF(*new_names)` | ALL columns at once | Must provide all names in order |
| `df.select(col('old').alias('new'), ...)` | Columns in select | Most flexible |

### withColumnRenamed Chaining
```python
df.withColumnRenamed('name',    'employee_name') \
  .withColumnRenamed('salary',  'annual_salary') \
  .withColumnRenamed('active',  'is_active')
```

### Renaming All Columns with toDF()
```python
# Must list ALL columns; order must match original DataFrame column order
df.toDF('emp_id', 'full_name', 'dept', 'annual_salary', 'is_active')
```

### drop() — Remove Columns
```python
df.drop('id', 'active')          # drop multiple columns
df.drop(col('id'), col('active')) # drop using Column objects
```

**Exam tip:** `withColumnRenamed` does NOT modify the original DataFrame (DataFrames are immutable).
It returns a new DataFrame with the renamed column.

In [ ]:
# Cell 4: Column renaming — all methods

print('=== withColumnRenamed — single column ===')
df_r1 = df_names.withColumnRenamed('name', 'employee_name') \
                .withColumnRenamed('salary', 'annual_salary')
df_r1.printSchema()
df_r1.show(3)

print('=== alias() inside select ===')
df_r2 = df_names.select(
    col('id'),
    col('name').alias('employee_name'),
    col('department').alias('dept'),
    col('salary').alias('annual_salary'),
    col('active').alias('is_active')
)
df_r2.show(3)

print('=== selectExpr — SQL AS syntax ===')
df_r3 = df_names.selectExpr(
    'id',
    'name AS employee_name',
    'department AS dept',
    'salary AS annual_salary',
    'active AS is_active'
)
df_r3.show(3)

print('=== toDF() — rename ALL columns at once ===')
df_r4 = df_names.toDF('emp_id', 'full_name', 'dept', 'annual_salary', 'is_active')
df_r4.printSchema()
df_r4.show(3)

print('=== drop() — remove columns ===')
df_dropped = df_names.drop('id', 'active')
df_dropped.printSchema()
df_dropped.show(3)

## selectExpr — SQL Expressions in DataFrame API

`selectExpr(*exprs)` accepts SQL expression strings and is equivalent to
`df.select(expr(col_expr_string))`.

```python
# These are all equivalent:
df.select(col('salary') * 1.1)
df.select(F.expr('salary * 1.1'))
df.selectExpr('salary * 1.1')
spark.sql('SELECT salary * 1.1 FROM tbl')

# selectExpr for complex transformations in a single call:
df.selectExpr(
    'name',
    'department AS dept',
    'salary * 1.1 AS salary_bumped',
    'CASE WHEN salary > 90000 THEN "senior" ELSE "junior" END AS grade',
    'upper(name) AS name_upper',
    'length(name) AS name_len',
    '* '   # include ALL original columns (only in pure selectExpr / spark.sql)
)
```

**Key selectExpr patterns tested on the exam:**

| Pattern | Example |
|---------|--------|
| Rename | `'old_name AS new_name'` |
| Compute | `'salary * 1.1 AS new_salary'` |
| Cast | `'CAST(salary AS STRING) AS salary_str'` |
| Conditional | `'CASE WHEN active THEN 1 ELSE 0 END AS active_int'` |
| Function | `'upper(name) AS name_upper'` |
| Wildcard | `'*'` (select all) |

In [ ]:
# Cell 5: selectExpr — SQL expression strings in DataFrame API

print('=== selectExpr — equivalent to SQL SELECT ===')
df_expr = df_names.selectExpr(
    'id',
    'name AS employee_name',
    'department AS dept',
    'salary * 1.1 AS salary_bumped',
    'CASE WHEN salary > 90000 THEN "senior" ELSE "junior" END AS grade',
    'upper(name) AS name_upper',
    'length(name) AS name_len'
)
df_expr.show()

print('=== selectExpr vs col() + F.expr() — all equivalent ===')

# selectExpr
a = df_names.selectExpr('salary * 1.1 AS new_salary')

# F.expr()
b = df_names.select(F.expr('salary * 1.1').alias('new_salary'))

# col() arithmetic
c = df_names.select((col('salary') * 1.1).alias('new_salary'))

print('selectExpr result:')
a.show(3)
print('F.expr() result (same):')
b.show(3)
print('col() arithmetic result (same):')
c.show(3)

# selectExpr with type cast
print('=== selectExpr CAST ===')
df_names.selectExpr(
    'name',
    'CAST(salary AS STRING) AS salary_str',
    'CAST(active AS INT) AS active_int'
).printSchema()

df_names.selectExpr(
    'name',
    'CAST(salary AS STRING) AS salary_str',
    'CAST(active AS INT) AS active_int'
).show(3)

In [ ]:
# Cell 6: Disambiguating columns after a join — df[col] vs col()

# Create two DataFrames with overlapping column names
employees = spark.createDataFrame([
    (1, 'Alice', 101),
    (2, 'Bob',   102),
    (3, 'Carol', 101),
], ['id', 'name', 'dept_id'])

departments = spark.createDataFrame([
    (101, 'Engineering', 'London'),
    (102, 'Marketing',   'New York'),
], ['id', 'dept_name', 'location'])  # 'id' clashes with employees.id!

joined = employees.join(departments, employees['dept_id'] == departments['id'], 'inner')

print('=== Schema after join — two id columns! ===')
joined.printSchema()
# Both 'id' columns exist — ambiguous reference

print('=== Disambiguate using df[col] ===')
result = joined.select(
    employees['id'].alias('emp_id'),
    employees['name'],
    departments['id'].alias('dept_id'),
    departments['dept_name'],
    departments['location']
)
result.show()

# This would raise AnalysisException (ambiguous):
try:
    joined.select(col('id')).show()
except Exception as e:
    print(f'Expected error: {type(e).__name__}: ambiguous column name')

# Clean join: rename BEFORE joining to avoid ambiguity
print('=== Best practice: rename before join ===')
clean_join = employees.join(
    departments.withColumnRenamed('id', 'dept_id_key'),
    employees['dept_id'] == departments['id'],
    'inner'
).drop('dept_id_key')
clean_join.show()

In [ ]:
# Cell 7: Schema inspection methods

print('=== df.printSchema() — tree view ===')
df_schema.printSchema()

print('=== df.schema — StructType object ===')
print(df_schema.schema)

print('\n=== df.schema.fields — list of StructFields ===')
for field in df_schema.schema.fields:
    print(f'  {field.name:15s} {str(field.dataType):20s} nullable={field.nullable}')

print('\n=== df.columns — list of column name strings ===')
print(df_schema.columns)

print('\n=== df.dtypes — list of (name, type_string) tuples ===')
print(df_schema.dtypes)

print('\n=== Number of columns ===')
print(f'Column count: {len(df_schema.columns)}')

# Programmatic schema construction
print('\n=== Build schema programmatically from df.columns ===')
# Useful when dynamically selecting subset of columns
selected_cols = ['name', 'salary', 'active']
df_subset = df_schema.select([col(c) for c in selected_cols])
df_subset.printSchema()

spark.stop()
print('Done.')

## Quick Reference Summary

### DataFrame Creation

```python
# Most common patterns:
spark.createDataFrame(list_of_tuples, ['col1', 'col2'])       # from list, infer types
spark.createDataFrame(list_of_tuples, schema)                 # from list, explicit schema
spark.range(start, end, step)                                  # generates 'id' LongType column
rdd.toDF(['col1', 'col2'])                                     # RDD → DataFrame
df.toDF('new1', 'new2', 'new3')                                # rename ALL columns
spark.read.parquet('/path'), spark.read.csv(), spark.read.json() # file-based
spark.table('db.table_name')                                   # catalog table
```

### Column Selection

```python
df.select('col1', 'col2')                 # strings
df.select(col('col1'), col('col2'))       # Column objects (supports expressions)
df.select(df['col1'], df['col2'])         # DataFrame-bound (safe after joins)
df.select(df.col1, df.col2)               # attribute (fails for spaces/dots)
df.selectExpr('col1', 'col2 AS c2', 'col1 * 2 AS doubled')  # SQL strings
```

### Column Renaming

```python
df.withColumnRenamed('old', 'new')        # one column
df.select(col('old').alias('new'), ...)   # in select
df.selectExpr('old AS new', ...)          # SQL AS in selectExpr
df.toDF('n1', 'n2', 'n3', ...)            # ALL columns
df.drop('col1', 'col2')                   # remove columns
```

### Schema Inspection

```python
df.printSchema()       # tree view with types and nullable
df.schema             # StructType object
df.schema.fields      # list of StructField
df.columns            # list of column name strings
df.dtypes             # list of (name, type_string) tuples
```

## Real-World Scenarios

<details>
<summary>Scenario 1: Creating a test DataFrame with exact schema to unit test a pipeline</summary>

**Situation:**
A data engineer is writing unit tests for a transformation function.
They need a small DataFrame with an exact schema (specific types, nullable flags)
to ensure the function behaves correctly and schema assertions pass.

**Code:**
```python
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType

# Explicit schema ensures the test catches type mismatches
schema = StructType([
    StructField('order_id',    IntegerType(), nullable=False),
    StructField('customer_id', IntegerType(), nullable=False),
    StructField('amount',      DoubleType(),  nullable=True),
    StructField('status',      StringType(),  nullable=True),
    StructField('is_refunded', BooleanType(), nullable=False),
])

test_data = [
    (1, 101, 99.99,  'shipped',   False),
    (2, 102, 249.99, 'delivered', True),
    (3, 101, None,   'pending',   False),  # null amount
]

test_df = spark.createDataFrame(test_data, schema)
test_df.printSchema()

# Apply transformation and assert schema
result = my_transform_function(test_df)
assert result.schema == expected_schema, 'Schema mismatch!'
assert result.count() == 2, 'Expected 2 rows after filtering refunded orders'
```

**Expected Outcome:**
Test DataFrame has exact types and nullable flags. Type mismatches surface immediately
rather than silently producing wrong results.

**Exam Sub-topic:** `spark.createDataFrame()` with `StructType`; schema definition
</details>

<details>
<summary>Scenario 2: Dynamically selecting and renaming columns from a wide table</summary>

**Situation:**
A raw data table has 200 columns. A pipeline needs to select 15 specific columns,
rename them to follow a standard naming convention (snake_case), and drop the rest.

**Code:**
```python
# Column mapping: raw_name → standard_name
column_mapping = {
    'CustomerID':    'customer_id',
    'OrderDate':     'order_date',
    'TotalAmount':   'total_amount',
    'ShipCountry':   'ship_country',
    'ProductCode':   'product_code',
    # ... 10 more
}

# Build select expression dynamically
select_exprs = [col(raw).alias(standard) for raw, standard in column_mapping.items()]

# Single select applies both column pruning and renaming
result = wide_df.select(select_exprs)

# Verify
print(result.columns)
# ['customer_id', 'order_date', 'total_amount', ...]
```

**Expected Outcome:**
Wide 200-column table reduced to 15 standard-named columns in one transformation.
Column pruning in the physical plan means only 15 columns are read from source.

**Exam Sub-topic:** `select()` with `col().alias()`; dynamic column selection
</details>

<details>
<summary>Scenario 3: Resolving ambiguous column names after a self-join</summary>

**Situation:**
A developer performs a self-join to find employee-manager pairs.
After the join, every column appears twice. They need to select specific columns
from each side without `AnalysisException`.

**Code:**
```python
employees = spark.createDataFrame([
    (1, 'Alice',  None),  # Alice has no manager
    (2, 'Bob',    1),
    (3, 'Carol',  1),
    (4, 'Dave',   2),
], ['emp_id', 'emp_name', 'manager_id'])

managers = employees.alias('mgr')
staff    = employees.alias('staff')

# Self-join: staff → manager
pairs = staff.join(
    managers,
    col('staff.manager_id') == col('mgr.emp_id'),
    how='left'
)

# Select using alias-qualified column names
result = pairs.select(
    col('staff.emp_id').alias('employee_id'),
    col('staff.emp_name').alias('employee'),
    col('mgr.emp_name').alias('manager')
)

result.show()
# employee_id | employee | manager
# 2           | Bob      | Alice
# 3           | Carol    | Alice
# 4           | Dave     | Bob
# 1           | Alice    | null
```

**Expected Outcome:**
Clean employee-manager pairs with no ambiguity errors.
DataFrame aliases (`alias('mgr')`, `alias('staff')`) used to qualify column names.

**Exam Sub-topic:** Column disambiguation after joins; DataFrame `alias()`; `col('alias.colname')`
</details>

<details>
<summary>Scenario 4: Using spark.range() to generate synthetic load test data</summary>

**Situation:**
A performance engineer needs 10 million synthetic rows to benchmark a new pipeline.
The data should include a sequential ID, random amounts, categories, and dates.

**Code:**
```python
from pyspark.sql.functions import rand, expr, lit, to_date, date_add

# Start with spark.range() — generates 10M rows very fast in parallel
N = 10_000_000

synthetic = spark.range(N) \
    .withColumn('amount',     (rand() * 1000 + 1).cast('decimal(10,2)')) \
    .withColumn('category',   expr("CASE WHEN id % 3 = 0 THEN 'Electronics'"
                                   "     WHEN id % 3 = 1 THEN 'Clothing'"
                                   "     ELSE 'Food' END")) \
    .withColumn('customer_id', (rand() * 10000).cast('int') + 1) \
    .withColumn('order_date',  date_add(lit('2024-01-01').cast('date'),
                                         (rand() * 365).cast('int')))

synthetic.show(5)
print(f'Total rows: {synthetic.count():,}')
```

**Expected Outcome:**
10 million rows generated in-memory without reading any files.
`spark.range()` is highly parallelized — each partition generates its own id range.

**Exam Sub-topic:** `spark.range()`; generating synthetic DataFrames
</details>

<details>
<summary>Scenario 5: Converting a legacy RDD pipeline to DataFrame API</summary>

**Situation:**
A legacy Spark 1.x pipeline uses RDDs. The team wants to migrate to the DataFrame API
for better performance (Catalyst optimizer, code generation) and SQL interoperability.

**Code:**
```python
# Legacy RDD pipeline
raw_rdd = sc.textFile('/data/sales.txt')
parsed_rdd = raw_rdd.map(lambda line: line.split(',')) \
                    .map(lambda f: (int(f[0]), f[1], float(f[2])))
filtered_rdd = parsed_rdd.filter(lambda r: r[2] > 100.0)

# Migrate to DataFrame — step 1: convert with schema
schema = StructType([
    StructField('order_id', IntegerType(), nullable=False),
    StructField('category', StringType(),  nullable=True),
    StructField('amount',   DoubleType(),  nullable=True),
])
df = spark.createDataFrame(filtered_rdd, schema)

# Or: rdd.toDF(['order_id', 'category', 'amount'])
df = filtered_rdd.toDF(['order_id', 'category', 'amount'])

# Step 2: use DataFrame API for rest of pipeline
result = df.groupBy('category').agg(F.sum('amount').alias('total'))
result.show()

# Benefit: Catalyst optimizer, predicate pushdown, code generation
# (RDD has none of these)
```

**Expected Outcome:**
Legacy RDD pipeline migrated to DataFrame API. The same logic now benefits from
Catalyst optimization, which RDDs bypass entirely.

**Exam Sub-topic:** `rdd.toDF()`; `spark.createDataFrame(rdd, schema)`; RDD → DataFrame migration
</details>

## Exam Cheat Sheet

| Concept | Key Exam Fact |
|---------|---------------|
| `spark.createDataFrame(data, schema)` | Most flexible creation; schema can be list of names, StructType, or DDL string |
| `spark.range(N)` | Creates one column named `id` of `LongType`; highly parallelized |
| `rdd.toDF(names)` | Converts RDD to DataFrame with given column names |
| `df.toDF(*names)` | Renames ALL columns; must provide same count as existing columns |
| `spark.table('name')` | Reads from Hive/Delta metastore |
| `'col_name'` in select | String — simple selection only, no expressions |
| `col('col_name')` | Column object — supports expressions, arithmetic, functions |
| `df['col_name']` | DataFrame-bound — unambiguous after joins |
| `df.col_name` | Attribute — fails for names with spaces or dots |
| `col('old').alias('new')` | Rename inside select |
| `df.withColumnRenamed('old','new')` | Rename single column; returns new DataFrame |
| `df.selectExpr('old AS new')` | SQL string rename in DataFrame API |
| `df.drop('col1', 'col2')` | Remove columns; accepts multiple |
| Backtick escaping | `col('\`first name\`')` — for spaces/dots/special chars |
| Ambiguous after join | Use `df['col']` or `col('alias.col')` — NOT `col('col')` |
| `df.columns` | List of column name strings |
| `df.dtypes` | List of `(name, type_string)` tuples |
| `df.schema` | `StructType` object |
| `df.printSchema()` | Prints tree view with types and nullable flags |

---

## Practice Q&A

<details>
<summary>Q1: What does spark.range(5) return?</summary>

**A:** A DataFrame with a single column named `id` of type `LongType`, containing rows 0, 1, 2, 3, 4 (5 rows). `spark.range(start, end, step)` — with one argument it is `range(end)`, starting from 0 with step 1. The column is always called `id` and is always `LongType`.
</details>

<details>
<summary>Q2: After joining two DataFrames that both have a column called 'id', how do you select each one?</summary>

**A:** Use DataFrame-bound column references: `df1['id']` and `df2['id']`. Using `col('id')` after the join raises `AnalysisException: Ambiguous reference to fields`. You can also use DataFrame aliases: create `df1.alias('a')` and `df2.alias('b')`, then reference `col('a.id')` and `col('b.id')`.
</details>

<details>
<summary>Q3: What is the difference between withColumnRenamed and toDF?</summary>

**A:**
- `withColumnRenamed('old', 'new')` — renames **one specific column** by name. All other columns are unchanged. Works even if you don't know the column positions.
- `df.toDF('n1', 'n2', 'n3', ...)` — renames **ALL columns at once** by position. You must provide exactly the same number of names as there are columns, in the correct order. Used for bulk rename when source column names don't match target convention.

Both return a **new DataFrame** — DataFrames are immutable.
</details>

<details>
<summary>Q4: How do you select a column whose name contains a space?</summary>

**A:** Use **backtick escaping**:
```python
df.select(col('`first name`'))          # using col()
df.selectExpr('`first name` AS fname')  # using selectExpr
spark.sql('SELECT `first name` FROM tbl')
```
The `df['first name']` subscript syntax also works without backticks:
```python
df.select(df['first name'])  # subscript handles spaces
```
But `df.first name` attribute syntax fails with a syntax error (space in Python identifier).
</details>

<details>
<summary>Q5: What is selectExpr and when would you choose it over select(col(...))?</summary>

**A:** `selectExpr(*exprs)` accepts **SQL expression strings** — it is equivalent to `df.select([expr(e) for e in exprs])`. Choose it when:
- You want to write SQL-style expressions inline: `'salary * 1.1 AS new_sal'`
- You're combining renaming + computation in one string: `'CASE WHEN active THEN 1 ELSE 0 END AS active_int'`
- You prefer SQL syntax for readability

Choose `select(col(...))` when:
- You're building column expressions programmatically in Python
- You need to pass `Column` objects from functions or variables
- You want IDE type checking on the expressions

Both produce identical physical plans after Catalyst optimization.
</details>